In [1]:
%pip install -U pip
%pip install pandas numpy scikit-learn tqdm

Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple/
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import random
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

import mindspore as ms
from mindspore import context, nn, ops, Tensor

from sklearn.metrics import roc_auc_score, log_loss

# ------------------------------
# Runtime configuration
# ------------------------------
DEVICE_TARGET = os.getenv("MS_DEVICE_TARGET", "GPU")  # GPU / Ascend / CPU
context.set_context(mode=context.GRAPH_MODE, device_target=DEVICE_TARGET)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
ms.set_seed(SEED)

# ------------------------------
# Canonical paths (must be in SAME directory as this notebook runtime cwd)
# ------------------------------
NOTEBOOK_DIR = Path.cwd()
TAXONOMY_PATH = NOTEBOOK_DIR / "cs_form3_form4_dkt_taxonomy.csv"
EVENTS_PATH = NOTEBOOK_DIR / "cs_dkt_events.csv"  # DB export in interaction_events-like shape

ARTIFACT_DIR = NOTEBOOK_DIR
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

SKILL_MAP_VERSION = "v1"
SKILL_MAP_PATH = ARTIFACT_DIR / f"skill_map_{SKILL_MAP_VERSION}.json"
MODEL_META_PATH = ARTIFACT_DIR / "model_meta.json"

assert TAXONOMY_PATH.exists(), f"Missing taxonomy file: {TAXONOMY_PATH}"
assert EVENTS_PATH.exists(), (
    f"Missing training events file: {EVENTS_PATH}\n"
    "Export DB events into this path using the agreed event schema first."
)

print("MindSpore:", ms.__version__)
print("Device:", ms.get_context("device_target"))
print("Taxonomy:", TAXONOMY_PATH)
print("Events:", EVENTS_PATH)
print("Output dir:", ARTIFACT_DIR)


MindSpore: 1.8.0
Device: GPU
Taxonomy: /workspace/workspace/dkt/update/cs_form3_form4_dkt_taxonomy.csv
Events: /workspace/workspace/dkt/update/cs_dkt_events.csv
Output dir: /workspace/workspace/dkt/update


Load DB-exported event data (interaction_events-style) and validate contract.

Required columns:
- student_id
- skill_code
- is_correct (0/1)
- event_time

Optional but useful:
- subject_code / subject_id
- score / max_score
- attempt_id / question_id



In [3]:
events = pd.read_csv(EVENTS_PATH)

required_cols = ["student_id", "skill_code", "is_correct", "event_time"]
missing = [c for c in required_cols if c not in events.columns]
assert not missing, f"Missing required columns in events CSV: {missing}"

# Normalize data types
for c in ["student_id", "skill_code", "event_time"]:
    events[c] = events[c].astype(str).str.strip()

events = events[(events["student_id"] != "") & (events["skill_code"] != "")]

# Correctness must be binary
if events["is_correct"].dtype == object:
    events["is_correct"] = events["is_correct"].astype(str).str.strip().str.lower().map(
        {"1": 1, "0": 0, "true": 1, "false": 0, "yes": 1, "no": 0}
    )

events["is_correct"] = pd.to_numeric(events["is_correct"], errors="coerce")
events = events.dropna(subset=["is_correct"])
events["is_correct"] = events["is_correct"].astype(int).clip(0, 1)

# Parse event time
events["event_time"] = pd.to_datetime(events["event_time"], errors="coerce", utc=True)
events = events.dropna(subset=["event_time"]).copy()

# Optional: keep only Computer Science if subject column exists
if "subject_code" in events.columns:
    events["subject_code"] = events["subject_code"].astype(str).str.strip().str.lower()
    events = events[events["subject_code"].isin(["computer_science", "cs", "comp_sci"]) | (events["subject_code"] == "")]

# Stable ordering keys for sequential modeling
sort_cols = ["student_id", "event_time"]
for extra in ["attempt_id", "question_id", "id"]:
    if extra in events.columns:
        sort_cols.append(extra)

events = events.sort_values(sort_cols).reset_index(drop=True)

print("Events rows after cleaning:", len(events))
print("Unique students:", events["student_id"].nunique())
print(events[["student_id", "skill_code", "is_correct", "event_time"]].head())


Events rows after cleaning: 61730
Unique students: 320
  student_id                                skill_code  is_correct  \
0   stu_0001         CS.F3.SAD.DESCRIBE_ANALYSIS_STAGE           0   
1   stu_0001            CS.F3.ALG.CONSTRUCT_PSEUDOCODE           0   
2   stu_0001             CS.F3.TECH.IDENTIFY_TECH_LAWS           1   
3   stu_0001  CS.F3.HW.IDENTIFY_HW_DEVICE_APPLICATIONS           0   
4   stu_0001          CS.F3.DATA.CONVERT_DENARY_TO_HEX           0   

                 event_time  
0 2025-02-14 12:22:00+00:00  
1 2025-02-14 14:37:00+00:00  
2 2025-02-14 17:24:00+00:00  
3 2025-02-14 18:19:00+00:00  
4 2025-02-14 18:42:00+00:00  


Freeze skill indices from syllabus taxonomy (do not rebuild from raw events).

- Build deterministic mapping once from taxonomy.
- Save mapping artifact per model version.
- Reuse same mapping for all future training/inference of that model version.



In [4]:
def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def save_json(path: Path, obj: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=True), encoding="utf-8")

# Load curriculum taxonomy
taxonomy = pd.read_csv(TAXONOMY_PATH)
required_tax_cols = ["skill_code", "dkt_trackable", "active"]
missing_tax = [c for c in required_tax_cols if c not in taxonomy.columns]
assert not missing_tax, f"Missing required taxonomy columns: {missing_tax}"

for c in ["skill_code"]:
    taxonomy[c] = taxonomy[c].astype(str).str.strip()

taxonomy = taxonomy[(taxonomy["active"] == 1) & (taxonomy["dkt_trackable"] == 1)]
taxonomy = taxonomy[taxonomy["skill_code"] != ""].copy()

canonical_skill_codes = sorted(taxonomy["skill_code"].unique().tolist())
assert canonical_skill_codes, "No active trackable skills found in taxonomy."

taxonomy_fingerprint = sha256_text("\n".join(canonical_skill_codes))

if SKILL_MAP_PATH.exists():
    saved_map = json.loads(SKILL_MAP_PATH.read_text(encoding="utf-8"))
    saved_codes = sorted(saved_map["skill_code_to_idx"].keys())

    if saved_codes != canonical_skill_codes:
        raise ValueError(
            "Frozen skill map does not match current taxonomy. "
            "Create a NEW mapping/model version instead of mutating indices."
        )

    skill_to_idx = {k: int(v) for k, v in saved_map["skill_code_to_idx"].items()}
    print(f"Loaded frozen mapping: {SKILL_MAP_PATH}")
else:
    skill_to_idx = {code: idx for idx, code in enumerate(canonical_skill_codes)}

    save_json(SKILL_MAP_PATH, {
        "mapping_version": SKILL_MAP_VERSION,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "taxonomy_path": str(TAXONOMY_PATH),
        "taxonomy_fingerprint": taxonomy_fingerprint,
        "num_skills": len(skill_to_idx),
        "skill_code_to_idx": skill_to_idx,
    })
    print(f"Created new frozen mapping: {SKILL_MAP_PATH}")

# Reverse mapping by index
idx_to_skill = [None] * len(skill_to_idx)
for code, idx in skill_to_idx.items():
    idx_to_skill[idx] = code

K = len(skill_to_idx)

# Validate events against frozen mapping
unknown_mask = ~events["skill_code"].isin(skill_to_idx)
unknown_n = int(unknown_mask.sum())
if unknown_n > 0:
    samples = events.loc[unknown_mask, "skill_code"].drop_duplicates().head(20).tolist()
    raise ValueError(
        f"Found {unknown_n} rows with unknown skill_code not in frozen taxonomy map. "
        f"Examples: {samples}"
    )

events["skill_idx"] = events["skill_code"].map(skill_to_idx).astype(int)

print("Num skills (K):", K)
print("First 10 skills:", idx_to_skill[:10])


Created new frozen mapping: /workspace/workspace/dkt/update/skill_map_v1.json
Num skills (K): 77
First 10 skills: ['CS.F3.ALG.APPLY_TOP_DOWN', 'CS.F3.ALG.CONSTRUCT_PSEUDOCODE', 'CS.F3.ALG.DEBUG_ALGORITHMS', 'CS.F3.ALG.DESIGN_FLOWCHARTS', 'CS.F3.ALG.USE_TRACE_TABLES', 'CS.F3.APP.DESCRIBE_APPLICATION_AREAS', 'CS.F3.DATA.CONVERT_DENARY_TO_HEX', 'CS.F3.DATA.CONVERT_DENARY_TO_OCTAL', 'CS.F3.DATA.CONVERT_HEX_TO_DENARY', 'CS.F3.DATA.CONVERT_OCTAL_TO_DENARY']


Build per-student ordered interaction sequences.
Each event is one question-level observation.



In [5]:
user_seqs = []
user_ids = []

for uid, g in events.groupby("student_id", sort=False):
    skills = g["skill_idx"].to_numpy(np.int32)
    corr = g["is_correct"].to_numpy(np.int32)

    # Need at least 3 events for next-step prediction signal
    if len(skills) < 3:
        continue

    user_ids.append(uid)
    user_seqs.append((skills, corr))

assert len(user_seqs) > 0, "No usable student sequences found."

lengths = np.array([len(s[0]) for s in user_seqs], dtype=np.int32)
print("Sequences kept:", len(user_seqs))
print("Min/Median/Max length:", int(lengths.min()), float(np.median(lengths)), int(lengths.max()))

# Train/Val/Test split by student (no leakage)
uids = np.array(user_ids)
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(uids))

n = len(uids)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]

train_seqs = [user_seqs[i] for i in train_idx]
val_seqs = [user_seqs[i] for i in val_idx]
test_seqs = [user_seqs[i] for i in test_idx]

print("Train/Val/Test users:", len(train_seqs), len(val_seqs), len(test_seqs))


Sequences kept: 320
Min/Median/Max length: 120 194.5 260
Train/Val/Test users: 256 32 32


Windowing + dataset generator (DKT standard).

Tokenization convention (must stay identical for train + inference):
- PAD token = 0
- interaction token = skill_idx + is_correct * K + 1
  (+1 offset reserves 0 strictly for PAD)



Targets per timestep:
- next_skill (shifted skills)
- next_correct (shifted correctness)
- mask (ignore PAD targets)



In [6]:
MAX_LEN = 50
STRIDE = 25
BATCH_SIZE = 32

PAD_X = 0
PAD_SKILL = -1

def make_windows(skills, corr, max_len, stride):
    L = len(skills)
    out = []
    start = 0
    while start < L:
        end = min(start + max_len, L)
        out.append((skills[start:end], corr[start:end]))
        if end == L:
            break
        start += stride
    return out

def pad_seq(arr, max_len, pad_val):
    out = np.full((max_len,), pad_val, dtype=np.int32)
    out[:len(arr)] = arr
    return out

def gen_samples(seqs, max_len):
    for skills, corr in seqs:
        for ws, wc in make_windows(skills, corr, max_len, STRIDE):
            if len(ws) < 2:
                continue

            # IMPORTANT: +1 offset to keep PAD=0 unique
            x_tok = (ws + wc * K + 1).astype(np.int32)
            x_tok = pad_seq(x_tok, max_len, pad_val=PAD_X)

            next_skill = pad_seq(ws[1:].astype(np.int32), max_len, pad_val=PAD_SKILL)
            next_corr = pad_seq(wc[1:].astype(np.int32), max_len, pad_val=0)
            mask = (next_skill != PAD_SKILL).astype(np.float32)

            yield x_tok, next_skill, next_corr, mask

tmp = next(gen_samples(train_seqs[:1], MAX_LEN))
print([a.shape for a in tmp], "mask sum:", float(tmp[-1].sum()))


[(50,), (50,), (50,), (50,)] mask sum: 49.0


In [7]:
import mindspore.dataset as ds

ds.config.set_num_parallel_workers(1)
ds.config.set_prefetch_size(2)
ds.config.set_enable_shared_mem(False)

def build_dataset(seqs, batch_size, max_len, shuffle=True):
    columns = ["x_tok", "next_skill", "next_corr", "mask"]
    dataset = ds.GeneratorDataset(
        source=lambda: gen_samples(seqs, max_len),
        column_names=columns,
        shuffle=shuffle,
        num_parallel_workers=1,
        python_multiprocessing=False,
    )
    # Keep remainder to avoid silent data loss
    dataset = dataset.batch(batch_size, drop_remainder=False)
    return dataset

train_ds = build_dataset(train_seqs, BATCH_SIZE, MAX_LEN, shuffle=True)
val_ds = build_dataset(val_seqs, BATCH_SIZE, MAX_LEN, shuffle=False)
test_ds = build_dataset(test_seqs, BATCH_SIZE, MAX_LEN, shuffle=False)

print("Train batches:", train_ds.get_dataset_size())


Train batches: 58


Standard DKT-LSTM model producing per-timestep per-skill logits: (B, T, K).



In [8]:
class DKTNetLSTM(nn.Cell):
    def __init__(self, num_skills, emb_dim=64, hidden_dim=128, dropout=0.0, num_layers=1):
        super().__init__()
        self.K = num_skills
        self.vocab = 2 * num_skills + 1  # +1 PAD token
        self.emb = nn.Embedding(self.vocab, emb_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            has_bias=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.fc = nn.Dense(hidden_dim, num_skills)

    def construct(self, x_tok):
        x = self.emb(x_tok)
        h, _ = self.lstm(x)
        logits = self.fc(h)
        return logits

model = DKTNetLSTM(K, emb_dim=64, hidden_dim=128, dropout=0.0, num_layers=1)
print(model)


DKTNetLSTM<
  (emb): Embedding<vocab_size=155, embedding_size=64, use_one_hot=False, embedding_table=Parameter (name=emb.embedding_table, shape=(155, 64), dtype=Float32, requires_grad=True), dtype=Float32, padding_idx=0>
  (lstm): LSTM<
    (rnn): _DynamicLSTMCPUGPU<>
    (dropout_op): Dropout<keep_prob=1.0>
    >
  (fc): Dense<input_channels=128, output_channels=77, has_bias=True>
  >


Masked next-skill BCE loss:
- p = sigmoid(logits[b,t,next_skill[b,t]])
- BCE vs next_correct
- average over valid (non-pad) targets only



In [9]:
sigmoid = ops.Sigmoid()
gather = ops.GatherD()
eps = 1e-7

class DKTLossCell(nn.Cell):
    def __init__(self, net):
        super().__init__()
        self.net = net

    def construct(self, x_tok, next_skill, next_corr, mask):
        logits = self.net(x_tok)
        probs = sigmoid(logits)

        safe_skill = ops.maximum(next_skill, Tensor(0, ms.int32))
        idx = ops.ExpandDims()(safe_skill, -1)
        p = gather(probs, 2, idx).squeeze(-1)

        y = next_corr.astype(ms.float32)
        p = ops.clip_by_value(p, eps, 1 - eps)

        bce = -(y * ops.log(p) + (1 - y) * ops.log(1 - p))
        loss = (bce * mask).sum() / (mask.sum() + eps)
        return loss

loss_cell = DKTLossCell(model)
optimizer = nn.Adam(model.trainable_params(), learning_rate=1e-3)
train_step = nn.TrainOneStepCell(loss_cell, optimizer)
train_step.set_train()


TrainOneStepCell<
  (network): DKTLossCell<
    (net): DKTNetLSTM<
      (emb): Embedding<vocab_size=155, embedding_size=64, use_one_hot=False, embedding_table=Parameter (name=net.emb.embedding_table, shape=(155, 64), dtype=Float32, requires_grad=True), dtype=Float32, padding_idx=0>
      (lstm): LSTM<
        (rnn): _DynamicLSTMCPUGPU<>
        (dropout_op): Dropout<keep_prob=1.0>
        >
      (fc): Dense<input_channels=128, output_channels=77, has_bias=True>
      >
    >
  (optimizer): Adam<>
  >

In [10]:
batch = next(train_ds.create_dict_iterator())
x_tok = batch["x_tok"]
print("x_tok shape:", x_tok.shape)

model.set_train(False)
_ = model(x_tok)
print("Forward pass OK")


x_tok shape: (32, 50)
Forward pass OK


Training + validation loop (AUC, accuracy, logloss).



In [11]:
def eval_epoch(net, dataset, max_batches=None):
    net.set_train(False)
    all_p, all_y = [], []

    for bi, batch in enumerate(dataset.create_dict_iterator()):
        if max_batches is not None and bi >= max_batches:
            break

        x_tok = batch["x_tok"]
        ns = batch["next_skill"].asnumpy().astype(np.int32)
        y = batch["next_corr"].asnumpy().astype(np.float32)
        m = batch["mask"].asnumpy().astype(np.float32)

        probs = sigmoid(net(x_tok)).asnumpy().astype(np.float32)
        ns_clip = np.clip(ns, 0, K - 1)
        p = np.take_along_axis(probs, ns_clip[..., None], axis=2).squeeze(2)

        valid = m > 0
        all_p.append(p[valid])
        all_y.append(y[valid])

    if not all_p:
        return {"auc": np.nan, "acc": np.nan, "logloss": np.nan}

    p = np.concatenate(all_p)
    y = np.concatenate(all_y)

    auc = roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan
    acc = float(((p >= 0.5) == (y >= 0.5)).mean())
    ll = float(log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)))

    return {"auc": float(auc), "acc": acc, "logloss": ll}


In [12]:
CKPT_DIR = ARTIFACT_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = CKPT_DIR / "best_dkt_lstm.ckpt"
print("Saving checkpoints to:", BEST_CKPT)

best_auc = -1.0
patience = 3
bad = 0
EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    model.set_train(True)
    losses = []

    for batch in tqdm(train_ds.create_dict_iterator(), total=train_ds.get_dataset_size(), desc=f"Epoch {epoch}"):
        loss = train_step(batch["x_tok"], batch["next_skill"], batch["next_corr"], batch["mask"])
        losses.append(float(loss.asnumpy()))

    val_metrics = eval_epoch(model, val_ds)
    print(
        f"\nEpoch {epoch} | train_loss={np.mean(losses):.4f} | "
        f"val_auc={val_metrics['auc']:.4f} | val_acc={val_metrics['acc']:.4f} | "
        f"val_logloss={val_metrics['logloss']:.4f}\n"
    )

    if val_metrics["auc"] > best_auc + 1e-4:
        best_auc = val_metrics["auc"]
        bad = 0
        ms.save_checkpoint(model, str(BEST_CKPT))
        print(f"Saved best checkpoint to {BEST_CKPT} (auc={best_auc:.4f})")
    else:
        bad += 1
        if bad >= patience:
            print(f"Early stopping (best auc={best_auc:.4f})")
            break

ms.load_checkpoint(str(BEST_CKPT), net=model)
test_metrics = eval_epoch(model, test_ds)
print("TEST(best):", test_metrics)

model_meta = {
    "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_type": "dkt_lstm",
    "skill_map_version": SKILL_MAP_VERSION,
    "skill_map_path": str(SKILL_MAP_PATH),
    "num_skills": K,
    "max_len": MAX_LEN,
    "stride": STRIDE,
    "batch_size": BATCH_SIZE,
    "checkpoint_path": str(BEST_CKPT),
    "metrics": test_metrics,
}
MODEL_META_PATH.write_text(json.dumps(model_meta, indent=2, ensure_ascii=True), encoding="utf-8")
print("Saved model metadata:", MODEL_META_PATH)


Saving checkpoints to: /workspace/workspace/dkt/update/checkpoints/best_dkt_lstm.ckpt


Epoch 1: 100%|███████████████████████████████████████████████████████████| 58/58 [00:01<00:00, 35.37it/s]



Epoch 1 | train_loss=0.6900 | val_auc=0.5194 | val_acc=0.5723 | val_logloss=0.6843

Saved best checkpoint to /workspace/workspace/dkt/update/checkpoints/best_dkt_lstm.ckpt (auc=0.5194)


Epoch 2: 100%|██████████████████████████████████████████████████████████| 58/58 [00:00<00:00, 275.30it/s]



Epoch 2 | train_loss=0.6862 | val_auc=0.5269 | val_acc=0.5735 | val_logloss=0.6822

Saved best checkpoint to /workspace/workspace/dkt/update/checkpoints/best_dkt_lstm.ckpt (auc=0.5269)


...


Epoch 4 | train_loss=0.6829 | val_auc=0.5283 | val_acc=0.5683 | val_logloss=0.6833



Epoch 5: 100%|██████████████████████████████████████████████████████████| 58/58 [00:00<00:00, 291.06it/s]



Epoch 5 | train_loss=0.6818 | val_auc=0.5282 | val_acc=0.5635 | val_logloss=0.6836



Epoch 6: 100%|██████████████████████████████████████████████████████████| 58/58 [00:00<00:00, 313.48it/s]



Epoch 6 | train_loss=0.6808 | val_auc=0.5286 | val_acc=0.5627 | val_logloss=0.6835

Early stopping (best auc=0.5311)
TEST(best): {'auc': 0.5342081644657319, 'acc': 0.5253187286766027, 'logloss': 0.6926169805797303}
Saved model metadata: /workspace/workspace/dkt/update/model_meta.json


Final test evaluation.



In [13]:
test_metrics = eval_epoch(model, test_ds)
print("TEST:", test_metrics)


TEST: {'auc': 0.5342081644657319, 'acc': 0.5253187286766027, 'logloss': 0.6926169805797303}


Inference helpers:
- predict_next_correct: P(correct on target skill)
- mastery_vector_from_history: per-skill mastery at current state

Both use the SAME frozen tokenization as training (+1 offset, PAD=0).



In [14]:
def _encode_history_tokens(history_skills, history_corr, max_len=MAX_LEN):
    hs = np.asarray(history_skills, dtype=np.int32)
    hc = np.asarray(history_corr, dtype=np.int32)

    assert hs.shape == hc.shape, "history_skills and history_corr must have same length"

    if len(hs) == 0:
        return np.zeros((1, max_len), dtype=np.int32), 0

    hs = hs[-max_len:]
    hc = hc[-max_len:]
    valid_len = len(hs)

    # IMPORTANT: same convention used during training
    x_tok = hs + hc * K + 1
    x_tok = np.pad(x_tok, (0, max_len - valid_len), constant_values=0)

    return x_tok.reshape(1, -1).astype(np.int32), valid_len


def predict_next_correct(net, history_skills, history_corr, next_skill_idx, max_len=MAX_LEN):
    net.set_train(False)
    x_tok_np, valid_len = _encode_history_tokens(history_skills, history_corr, max_len=max_len)

    if valid_len == 0:
        return 0.5

    x_tok = Tensor(x_tok_np, ms.int32)
    probs = sigmoid(net(x_tok)).asnumpy()

    t = valid_len - 1  # last non-pad timestep
    return float(probs[0, t, int(next_skill_idx)])


def mastery_vector_from_history(net, history_skills, history_corr, max_len=MAX_LEN):
    net.set_train(False)
    x_tok_np, valid_len = _encode_history_tokens(history_skills, history_corr, max_len=max_len)

    if valid_len == 0:
        return np.full((K,), 0.5, dtype=np.float32)

    x_tok = Tensor(x_tok_np, ms.int32)
    probs = sigmoid(net(x_tok)).asnumpy()

    t = valid_len - 1
    return probs[0, t, :].astype(np.float32)


# Example quick check with first test sequence
if len(test_seqs) > 0:
    skills, corr = test_seqs[0]
    p = predict_next_correct(model, skills[:-1], corr[:-1], int(skills[-1]))
    print("Predicted P(correct next):", p, "| true:", int(corr[-1]))


Predicted P(correct next): 0.4676845073699951 | true: 0


In [15]:
# Service-style response example (top weak skills)
if len(test_seqs) > 0:
    skills, corr = test_seqs[0]
    mastery = mastery_vector_from_history(model, skills, corr)

    weak_idx = np.argsort(mastery)[:5]
    weak = [
        {
            "skill_code": idx_to_skill[int(i)],
            "mastery_prob": float(mastery[int(i)]),
        }
        for i in weak_idx
    ]

    avg_mastery = float(mastery.mean())
    risk_level = "high" if avg_mastery < 0.45 else ("medium" if avg_mastery < 0.70 else "low")

    dkt_response = {
        "student_id": "demo_student",
        "snapshot_time_utc": datetime.now(timezone.utc).isoformat(),
        "average_mastery": avg_mastery,
        "risk_level": risk_level,
        "weak_skills": weak,
        "model_meta_path": str(MODEL_META_PATH),
    }

    print(json.dumps(dkt_response, indent=2))


{
  "student_id": "demo_student",
  "snapshot_time_utc": "2026-03-05T12:00:57.326237+00:00",
  "average_mastery": 0.432098925113678,
  "risk_level": "high",
  "weak_skills": [
    {
      "skill_code": "CS.F3.DB.APPLY_DATABASE_SECURITY",
      "mastery_prob": 0.35726895928382874
    },
    {
      "skill_code": "CS.F4.DB.CREATE_ADVANCED_QUERIES",
      "mastery_prob": 0.3649512827396393
    },
    {
      "skill_code": "CS.F4.DATA.MODEL_LOGIC_CIRCUITS",
      "mastery_prob": 0.3829815089702606
    },
    {
      "skill_code": "CS.F3.DB.CREATE_RELATIONAL_DATABASES",
      "mastery_prob": 0.39257270097732544
    },
    {
      "skill_code": "CS.F4.APP.DESIGN_DOMAIN_MODELS",
      "mastery_prob": 0.393085777759552
    }
  ],
  "model_meta_path": "/workspace/workspace/dkt/update/model_meta.json"
}
